# NB8 — System Preference & IAA Analysis

In [ ]:
import os
if not os.path.ismount("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")

In [ ]:
import pandas as pd
import numpy as np
from itertools import combinations

MASTER_PATH = "Master_Eval_Sheet.xlsx"
OUTPUT_DIR  = "outputs/tables/"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Ready.")

Ready.


In [ ]:
col_names = [
    "#","Sentence_ID","Source_File","Source_Sentence","Gold_Category",
    "LLM","Has_Errors","Error_Span","Annotated_Category","Description","Corrected_Sentence",
    "H1_C1","H1_C2","H1_C3","H1_C4","H1_Total",
    "H2_C1","H2_C2","H2_C3","H2_C4","H2_Total",
    "L1_C1","L1_C2","L1_C3","L1_C4","L1_Total",
    "L2_C1","L2_C2","L2_C3","L2_C4","L2_Total",
    "L3_C1","L3_C2","L3_C3","L3_C4","L3_Total",
    "L4_C1","L4_C2","L4_C3","L4_C4","L4_Total",
]
df = pd.read_excel(MASTER_PATH, sheet_name="Master Eval Sheet",
                   header=None, skiprows=2, names=col_names)
df = df[df["#"].notna()].copy()
df = df[df["#"].astype(str) != "TOTAL"].reset_index(drop=True)
sarvam = df[df["LLM"]=="Sarvam"].set_index("Sentence_ID").copy()
llama  = df[df["LLM"]=="Llama"].set_index("Sentence_ID").copy()
sentences = sorted(sarvam.index.tolist())
print(f"Loaded: {len(sentences)} sentences, "
      f"{len(df[df['LLM']=='Sarvam'])} Sarvam rows, "
      f"{len(df[df['LLM']=='Llama'])} LLaMA rows")

Loaded: 50 sentences, 50 Sarvam rows, 50 LLaMA rows


In [ ]:
RATERS = ["H1","H2","L1","L2","L3","L4"]
# Updated to include sub-criteria AND the holistic Total
TASKS  = ["C1","C2","C3","C4","Total"]
CATEGORIES = [
    "Script Normalization",
    "Spelling & Typographical Error",
    "Grammatical Error",
    "Code-Mixing / Wrong Language",
    "Correct Sentence / No Errors",
]
CAT_SHORT = {
    "Script Normalization":           "Script Norm.",
    "Spelling & Typographical Error":  "Spelling",
    "Grammatical Error":              "Grammatical",
    "Code-Mixing / Wrong Language":   "Code-Mixing",
    "Correct Sentence / No Errors":   "Correct",
}
RATER_LABELS = {
    "H1": "Human 1",
    "H2": "Human 2",
    "L1": "Sarvam-105B",
    "L2": "LLaMA-4-Scout-17B",
    "L3": "GPT-Oss-120B",
    "L4": "Gemini-3.1-Flash-Lite-Preview",
}
TASK_LABELS = {
    "C1": "Detection (C1)",
    "C2": "Span+Description (C2)",
    "C3": "Category (C3)",
    "C4": "Correction (C4)",
    "Total": "Holistic Total (Out of 6)",
}
# Rater subgroups for multi-rater analysis
GROUPS = {
    "G1 — Humans":                      ["H1","H2"],
    "G2 — Annotator LLMs":              ["L1","L2"],
    "G3 — Non-Annotator LLMs":          ["L3","L4"],
    "G4 — All LLMs":                    ["L1","L2","L3","L4"],
    "G5 — Humans + Annotator LLMs":     ["H1","H2","L1","L2"],
    "G6 — Humans + Non-Annotator LLMs": ["H1","H2","L3","L4"],
    "G7 — All Raters":                  ["H1","H2","L1","L2","L3","L4"],
}
GROUP_ORDER = [
    "G1 — Humans",
    "G2 — Annotator LLMs",
    "G3 — Non-Annotator LLMs",
    "G4 — All LLMs",
    "G5 — Humans + Annotator LLMs",
    "G6 — Humans + Non-Annotator LLMs",
    "G7 — All Raters",
]
print("Constants defined.")

Constants defined.


In [ ]:
# ── IAA metric implementations ────────────────────────────────
# All work on nominal S/L/T labels, no external libraries needed.

def cohen_kappa(a, b):
    """Cohen's kappa for two lists of nominal labels."""
    a=list(a); b=list(b); n=len(a)
    cats=list(set(a+b))
    po=sum(1 for i in range(n) if a[i]==b[i])/n
    pe=sum((a.count(c)/n)*(b.count(c)/n) for c in cats)
    return (po-pe)/(1-pe) if pe<1 else 1.0

def fleiss_kappa(arr, cats=["S","L","T"]):
    """Fleiss' kappa for N_items x N_raters array of nominal labels."""
    N,R=arr.shape
    mat=np.zeros((N,len(cats)))
    for ci,c in enumerate(cats): mat[:,ci]=(arr==c).sum(axis=1)
    n=R; p_j=mat.sum(axis=0)/(N*n)
    P_i=((mat**2).sum(axis=1)-n)/(n*(n-1))
    P_bar=P_i.mean(); P_e=(p_j**2).sum()
    return (P_bar-P_e)/(1-P_e) if P_e<1 else 1.0

def kripp_alpha_nominal(arr, cats=["S","L","T"]):
    """Krippendorff's alpha (nominal) for N_items x N_raters label array."""
    N,R=arr.shape; coinc=[]
    for i in range(N):
        row=list(arr[i])
        for v1,v2 in combinations(row,2): coinc.append((v1,v2))
    if not coinc: return np.nan
    Do=sum(1 for v1,v2 in coinc if v1!=v2)/len(coinc)
    all_v=[v for row in arr for v in row]; n=len(all_v)
    pe=1-sum((all_v.count(c)/n)**2 for c in cats)
    return 1-Do/pe if pe>0 else np.nan

def gwet_ac1(arr, cats=["S","L","T"]):
    """Gwet's AC1 (nominal). Resistant to kappa paradox when one category dominates."""
    N,R=arr.shape; q=len(cats); agree=0; total=0
    for i in range(N):
        row=list(arr[i]); m=len(row)
        if m<2: continue
        n_agree=sum(1 for a,b in combinations(row,2) if a==b)
        agree+=n_agree/(m*(m-1)/2); total+=1
    po=agree/total if total>0 else 0
    all_v=[v for row in arr for v in row]; n=len(all_v)
    pe=sum((all_v.count(c)/n)*(1-all_v.count(c)/n) for c in cats)/(q-1) if q>1 else 0
    return (po-pe)/(1-pe) if pe<1 else 1.0

print("Metric functions defined.")

Metric functions defined.


## Part A: Item-Level S/L/T Preference Labels

In [ ]:
# ── Part A: Build item-level S/L/T preference labels ──────────
# For each sentence x task x rater:
#   S  = Sarvam score > LLaMA score (rater prefers Sarvam)
#   L  = LLaMA score  > Sarvam score (rater prefers LLaMA)
#   T  = scores equal (Tie)
# Human Gold label = majority vote of H1 + H2
#   If H1==H2 → use that label
#   If one is T → use the other
#   If H1≠H2 and neither is T → T (genuine disagreement)
rows=[]
for sid in sentences:
    row={"Sentence_ID":sid,
         "Gold_Category":sarvam.loc[sid,"Gold_Category"],
         "Cat_Short":CAT_SHORT[sarvam.loc[sid,"Gold_Category"]]}
    for task in TASKS:
        for rater in RATERS:
            s=sarvam.loc[sid,f"{rater}_{task}"]
            l=llama.loc[sid,f"{rater}_{task}"]
            row[f"{rater}_{task}_label"]="S" if s>l else "L" if l>s else "T"
        h1=row[f"H1_{task}_label"]; h2=row[f"H2_{task}_label"]
        if h1==h2:    gold=h1
        elif h1=="T": gold=h2
        elif h2=="T": gold=h1
        else:         gold="T"
        row[f"Gold_{task}"]=gold
    rows.append(row)
item_df=pd.DataFrame(rows)
print("Item-level label table:", item_df.shape)

print("\nLabel distribution (C1):")
for rater in RATERS:
    vc=item_df[f"{rater}_C1_label"].value_counts()
    print(f"  {RATER_LABELS[rater]:35s}: S={vc.get('S',0):3d} L={vc.get('L',0):3d} T={vc.get('T',0):3d}")

# Save item-level labels
cols=["Sentence_ID","Gold_Category","Cat_Short"]
for task in TASKS:
    for rater in RATERS:
        cols.append(f"{rater}_{task}_label")
    cols.append(f"Gold_{task}")
item_df[cols].to_excel(OUTPUT_DIR+"NB8_Item_Labels.xlsx", index=False)
print("Saved: NB8_Item_Labels.xlsx")

Item-level label table: (50, 38)

Label distribution (C1):
  Human 1                            : S= 11 L=  7 T= 32
  Human 2                            : S= 11 L=  7 T= 32
  Sarvam-105B                        : S= 14 L=  7 T= 29
  LLaMA-4-Scout-17B                  : S=  0 L=  3 T= 47
  GPT-Oss-120B                       : S= 18 L=  7 T= 25
  Gemini-3.1-Flash-Lite-Preview      : S= 11 L=  7 T= 32
Saved: NB8_Item_Labels.xlsx


## Part B: Raw Agreement Rate with Human Gold

In [ ]:
# ── Part B: Raw Agreement Rate ────────────────────────────────
# For each rater x task x category:
# % of sentences where rater's S/L/T label == Human Gold label
agree_rows=[]
for rater in RATERS:
    for task in TASKS:
        gold_col=f"Gold_{task}"
        rater_col=f"{rater}_{task}_label"
        # Overall (all 50 sentences)
        agrees=int((item_df[rater_col]==item_df[gold_col]).sum())
        total=len(item_df)
        agree_rows.append({
            "Rater":rater,"Rater_Label":RATER_LABELS[rater],
            "Task":task,"Task_Label":TASK_LABELS[task],
            "Category":"ALL","Cat_Short":"ALL",
            "Agreements":agrees,"Total":total,
            "Agreement_Rate":round(agrees/total,4)
        })
        # Per category
        for cat in CATEGORIES:
            sub=item_df[item_df["Gold_Category"]==cat]
            agr=int((sub[rater_col]==sub[gold_col]).sum())
            tot=len(sub)
            agree_rows.append({
                "Rater":rater,"Rater_Label":RATER_LABELS[rater],
                "Task":task,"Task_Label":TASK_LABELS[task],
                "Category":cat,"Cat_Short":CAT_SHORT[cat],
                "Agreements":agr,"Total":tot,
                "Agreement_Rate":round(agr/tot,4) if tot>0 else np.nan
            })
agree_df=pd.DataFrame(agree_rows)
agree_df.to_excel(OUTPUT_DIR+"NB8_Agreement_Rate.xlsx", index=False)
print("Saved: NB8_Agreement_Rate.xlsx")

# Summary pivot
print("\nOverall agreement rate (all categories combined):")
ov=agree_df[agree_df["Cat_Short"]=="ALL"]
piv=ov.pivot(index="Rater_Label",columns="Task_Label",values="Agreement_Rate")
RATER_ORDER=["Human 1","Human 2","Sarvam-105B","LLaMA-4-Scout-17B","GPT-Oss-120B","Gemini-3.1-Flash-Lite-Preview"]
TASK_ORDER =["Detection (C1)","Span+Description (C2)","Category (C3)","Correction (C4)","Holistic Total (Out of 6)"]
piv=piv.reindex(index=RATER_ORDER,columns=TASK_ORDER)
print(piv.to_string())

Saved: NB8_Agreement_Rate.xlsx

Overall agreement rate (all categories combined):
Task_Label                     Detection (C1)  Span+Description (C2)  Category (C3)  Correction (C4)  Holistic Total (Out of 6)
Rater_Label                                                                                                                    
Human 1                                  0.96                   0.98           0.96             0.92                       0.92
Human 2                                  0.96                   0.96           0.98             0.92                       0.96
Sarvam-105B                              0.54                   0.54           0.54             0.54                       0.56
LLaMA-4-Scout-17B                        0.58                   0.58           0.46             0.40                       0.50
GPT-Oss-120B                             0.56                   0.52           0.54             0.48                       0.64
Gemini-3.1-Flash-Lite-

## Part C: Optimism Bias (Self-Evaluating LLMs)

In [ ]:
# ── Part C: Optimism Bias ─────────────────────────────────────
# Only for self-evaluating LLMs: L1 (Sarvam) and L2 (LLaMA)
# Bias = evaluator's score_diff minus human baseline score_diff
# score_diff = mean(Sarvam scores) - mean(LLaMA scores) for a task/category
# Positive bias means the self-evaluator is MORE lenient toward its own output
# than humans are (optimism = overrating self).
bias_rows=[]
for task in TASKS:
    for cat in CATEGORIES:
        cat_s=sarvam[sarvam["Gold_Category"]==cat]
        cat_l=llama[llama["Gold_Category"]==cat]
        # Human baseline: average of H1 and H2 score differences
        h_s=(cat_s[f"H1_{task}"].mean()+cat_s[f"H2_{task}"].mean())/2
        h_l=(cat_l[f"H1_{task}"].mean()+cat_l[f"H2_{task}"].mean())/2
        human_diff=round(h_s-h_l,4)
        # L1 (Sarvam evaluates itself)
        ev_s_L1=cat_s[f"L1_{task}"].mean(); ev_l_L1=cat_l[f"L1_{task}"].mean()
        L1_diff=round(ev_s_L1-ev_l_L1,4)
        L1_bias=round(L1_diff-human_diff,4)
        bias_rows.append({"Rater_Code":"L1","Rater":"Sarvam-105B",
                          "Task":task,"Task_Label":TASK_LABELS[task],
                          "Category":cat,"Cat_Short":CAT_SHORT[cat],
                          "Human_Diff":human_diff,"Self_Diff":L1_diff,
                          "Optimism_Bias":L1_bias})
        # L2 (LLaMA evaluates itself) — bias toward LLaMA = -(ev_diff) - (-human_diff)
        ev_s_L2=cat_s[f"L2_{task}"].mean(); ev_l_L2=cat_l[f"L2_{task}"].mean()
        L2_diff=round(ev_s_L2-ev_l_L2,4)
        L2_bias=round((-L2_diff)-(-human_diff),4)
        bias_rows.append({"Rater_Code":"L2","Rater":"LLaMA-4-Scout-17B",
                          "Task":task,"Task_Label":TASK_LABELS[task],
                          "Category":cat,"Cat_Short":CAT_SHORT[cat],
                          "Human_Diff":human_diff,"Self_Diff":L2_diff,
                          "Optimism_Bias":L2_bias})
bias_df=pd.DataFrame(bias_rows)
bias_df.to_excel(OUTPUT_DIR+"NB8_Optimism_Bias.xlsx", index=False)
print("Saved: NB8_Optimism_Bias.xlsx")

print("\nOptimism bias (mean per rater per task):")
ob=bias_df.groupby(["Rater","Task_Label"])["Optimism_Bias"].mean().unstack().round(3)
print(ob.reindex(columns=TASK_ORDER).to_string())

Saved: NB8_Optimism_Bias.xlsx

Optimism bias (mean per rater per task):
Task_Label         Detection (C1)  Span+Description (C2)  Category (C3)  Correction (C4)  Holistic Total (Out of 6)
Rater                                                                                                              
LLaMA-4-Scout-17B            0.14                   0.16           0.27             0.58                       1.15
Sarvam-105B                  0.06                   0.24          -0.01             0.04                       0.33


Fleiss' κ, Krippendorff's α (nominal), Gwet's AC1 on the 50-sentence × N-rater S/L/T label matrix. Gwet's AC1 is preferred when one
category (Tie) dominates, as it is robust to the κ paradox.

## Part D: Multi-Rater Chance-Corrected Agreement

In [ ]:
# ── Part D: Multi-Rater Chance-Corrected Agreement ────────────
# Fleiss' kappa, Krippendorff's alpha (nominal), Gwet's AC1
# Applied to 50-sentence S/L/T label matrix per group per task
multi_rows=[]
for group_name, group_raters in GROUPS.items():
    for task in TASKS:
        cols=[f"{r}_{task}_label" for r in group_raters]
        arr=item_df[cols].values
        fk=round(fleiss_kappa(arr),4)
        ka=round(kripp_alpha_nominal(arr),4)
        ac=round(gwet_ac1(arr),4)
        multi_rows.append({
            "Group":group_name,"Task":task,"Task_Label":TASK_LABELS[task],
            "Category":pd.NA,"Cat_Short":pd.NA,
            "Fleiss_Kappa":fk,"Kripp_Alpha_Nominal":ka,"Gwet_AC1":ac
        })
        for cat in CATEGORIES:
            sub=item_df[item_df["Gold_Category"]==cat]
            if len(sub)<2: continue
            arr_c=sub[cols].values
            multi_rows.append({
                "Group":group_name,"Task":task,"Task_Label":TASK_LABELS[task],
                "Category":cat,"Cat_Short":CAT_SHORT[cat],
                "Fleiss_Kappa":round(fleiss_kappa(arr_c),4),
                "Kripp_Alpha_Nominal":round(kripp_alpha_nominal(arr_c),4),
                "Gwet_AC1":round(gwet_ac1(arr_c),4)
            })
multi_df=pd.DataFrame(multi_rows)
multi_df.to_excel(OUTPUT_DIR+"NB8_Multi_Rater_Metrics.xlsx", index=False)
print("Saved: NB8_Multi_Rater_Metrics.xlsx")

print("\nOverall multi-rater metrics (G7 — All Raters, no category filter):")
ov=multi_df[(multi_df["Group"]=="G7 — All Raters")&(multi_df["Cat_Short"].isna())]
print(ov[["Task_Label","Fleiss_Kappa","Kripp_Alpha_Nominal","Gwet_AC1"]].to_string(index=False))

Saved: NB8_Multi_Rater_Metrics.xlsx

Overall multi-rater metrics (G7 — All Raters, no category filter):
               Task_Label  Fleiss_Kappa  Kripp_Alpha_Nominal  Gwet_AC1
           Detection (C1)        0.1960               0.1960    0.4557
    Span+Description (C2)        0.3262               0.3262    0.4010
            Category (C3)        0.2530               0.2530    0.4054
          Correction (C4)        0.1894               0.1894    0.3911
Holistic Total (Out of 6)        0.3389               0.3389    0.3759


## Part E: Pairwise Cohen's κ (6×6 Matrix)

In [ ]:
# ── Part E: Pairwise Cohen's Kappa (6x6 matrix) ───────────────
# For every pair of raters, compute Cohen's kappa on S/L/T labels
# across 50 sentences. This gives a 6x6 symmetric matrix per task.
pair_rows=[]
for task in TASKS:
    for r1,r2 in combinations(RATERS,2):
        a=item_df[f"{r1}_{task}_label"].tolist()
        b=item_df[f"{r2}_{task}_label"].tolist()
        k=round(cohen_kappa(a,b),4)
        pair_rows.append({
            "Task":task,"Task_Label":TASK_LABELS[task],
            "Rater1":r1,"Rater2":r2,
            "Rater1_Label":RATER_LABELS[r1],"Rater2_Label":RATER_LABELS[r2],
            "Category":pd.NA,"Cat_Short":pd.NA,
            "Cohen_Kappa":k
        })
        for cat in CATEGORIES:
            sub=item_df[item_df["Gold_Category"]==cat]
            k_c=round(cohen_kappa(
                sub[f"{r1}_{task}_label"].tolist(),
                sub[f"{r2}_{task}_label"].tolist()),4)
            pair_rows.append({
                "Task":task,"Task_Label":TASK_LABELS[task],
                "Rater1":r1,"Rater2":r2,
                "Rater1_Label":RATER_LABELS[r1],"Rater2_Label":RATER_LABELS[r2],
                "Category":cat,"Cat_Short":CAT_SHORT[cat],
                "Cohen_Kappa":k_c
            })
pair_df=pd.DataFrame(pair_rows)
pair_df.to_excel(OUTPUT_DIR+"NB8_Pairwise_Cohen_Kappa.xlsx", index=False)
print("Saved: NB8_Pairwise_Cohen_Kappa.xlsx")

# Print 6x6 matrix for C1 as verification
print("\nCohen's \u03ba matrix (C1, all 50 sentences):")
rn=[RATER_LABELS[r] for r in RATERS]
mat=pd.DataFrame(np.nan,index=rn,columns=rn)
for r1,r2 in combinations(RATERS,2):
    k=pair_df[(pair_df["Task"]=="C1")&(pair_df["Rater1"]==r1)&
              (pair_df["Rater2"]==r2)&(pair_df["Cat_Short"].isna())]["Cohen_Kappa"].values[0]
    mat.loc[RATER_LABELS[r1],RATER_LABELS[r2]]=k
    mat.loc[RATER_LABELS[r2],RATER_LABELS[r1]]=k
for r in rn: mat.loc[r,r]=1.0
print(mat.round(3).to_string())

Saved: NB8_Pairwise_Cohen_Kappa.xlsx

Cohen's κ matrix (C1, all 50 sentences):
                               Human 1  Human 2  Sarvam-105B  LLaMA-4-Scout-17B  GPT-Oss-120B  Gemini-3.1-Flash-Lite-Preview
Human 1                          1.000    0.847        0.123              0.026         0.243                          0.196
Human 2                          0.847    1.000        0.087              0.026         0.174                          0.273
Sarvam-105B                      0.123    0.087        1.000              0.014         0.288                          0.379
LLaMA-4-Scout-17B                0.026    0.026        0.014              1.000         0.003                         -0.077
GPT-Oss-120B                     0.243    0.174        0.288              0.003         1.000                          0.277
Gemini-3.1-Flash-Lite-Preview    0.196    0.273        0.379             -0.077         0.277                          1.000


## Part F: Printable Preference Tables

In [ ]:
# ── Part F: Printable Preference Tables (one per task) ─────────
# Each table shows all 50 sentences, all 6 raters, Gold winner
print("Saving printable preference table...")
pref_rows=[]
for _, row_data in item_df.iterrows():
    sid=row_data["Sentence_ID"]; cat=row_data["Gold_Category"]
    pref_row={"Sentence_ID":sid,"Category":cat,"Cat_Short":CAT_SHORT[cat]}
    for task in TASKS:
        for rater in RATERS:
            pref_row[f"{RATER_LABELS[rater]}_{task}"]=row_data[f"{rater}_{task}_label"]
        pref_row[f"Gold_{task}"]=row_data[f"Gold_{task}"]
    pref_rows.append(pref_row)
pref_df=pd.DataFrame(pref_rows)
pref_df.to_excel(OUTPUT_DIR+"NB8_Preference_Table.xlsx", index=False)
print("Saved: NB8_Preference_Table.xlsx")

# Print one task as example
print("\nPreference table preview — C1 (first 10 sentences):")
preview_cols=["Sentence_ID","Cat_Short"] + \
    [f"{RATER_LABELS[r]}_C1" for r in RATERS] + ["Gold_C1"]
print(pref_df[preview_cols].head(10).to_string(index=False))
print("\nS=Sarvam wins, L=LLaMA wins, T=Tie")

Saving printable preference table...
Saved: NB8_Preference_Table.xlsx

Preference table preview — C1 (first 10 sentences):
Sentence_ID Cat_Short Human 1_C1 Human 2_C1 Sarvam-105B_C1 LLaMA-4-Scout-17B_C1 GPT-Oss-120B_C1 Gemini-3.1-Flash-Lite-Preview_C1 Gold_C1
      CS_01   Correct          T          T              T                    T               T                                T       T
      CS_02   Correct          L          L              L                    T               L                                L       L
      CS_03   Correct          S          S              S                    T               S                                S       S
      CS_04   Correct          T          T              L                    T               S                                T       T
      CS_05   Correct          T          T              T                    T               T                                S       T
      CS_06   Correct          T          T            